# CMPE 256 — Song Recommender System
## Starter Notebook

This notebook gives you everything you need to **get started** with the Kaggle competition:
- How to load the data files
- How to build and validate a prediction
- How to generate, download and submit a valid submission file

The model used here is the **Global Mean baseline**, the simplest possible predictor.  
Your job is to beat it.

---
> **Note:** EDA (Exploratory Data Analysis) is intentionally not included here. You are encouraged to explore the data yourself, understanding it is part of the assignment.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

# Load KAGGLE_USERNAME and KAGGLE_KEY from the .env file in this directory
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath("RS_HW2.ipynb")), ".env"))

DATA_DIR = os.path.abspath("/home/anthony/repos/cmpe256/data/raw") + "/" 
# os.path.join(os.path.dirname(os.path.abspath("/home/anthony/repos/cmpe256/data/raw")), "") + "/"
os.makedirs(DATA_DIR, exist_ok=True)

!pip install kaggle python-dotenv -q
!kaggle competitions download -c cmpe-256-song-recommender-system33 -p {DATA_DIR}
!unzip -q -o {DATA_DIR}/*.zip -d {DATA_DIR}


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


## Step 1 — Install & Import Libraries

In [2]:
# !pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from collections import defaultdict
from collections.abc import Callable 

print('Libraries ready.')

Libraries ready.


## Step 2 — Load the Data

Four files are provided:

| File | Description |
|------|-------------|
| `train.csv` | User-song pairs with ratings (1.0–10.0). Use this to train your model. |
| `test.csv` | User-song pairs **without** ratings. You must predict these. |
| `sample_submission.csv` | Shows the exact format your submission must follow. |
| `song_data.csv` | Optional song metadata (title, artist, album, year). Useful for hybrid models. |

In [3]:
print(f'Using data directory: {DATA_DIR}')
train      = pd.read_csv(DATA_DIR + 'train.csv',             encoding='latin-1')
test       = pd.read_csv(DATA_DIR + 'test.csv',              encoding='latin-1')
sample_sub = pd.read_csv(DATA_DIR + 'sample_submission.csv', encoding='latin-1')
song_data  = pd.read_csv(DATA_DIR + 'song_data.csv',         encoding='latin-1')

print('=== train.csv ===')
print(f'  Rows   : {len(train):,}')
print(f'  Columns: {list(train.columns)}')
display(train.head(3))

print('\n=== test.csv ===')
print(f'  Rows   : {len(test):,}')
print(f'  Columns: {list(test.columns)}')
display(test.head(3))

print('\n=== sample_submission.csv ===')
print(f'  Rows   : {len(sample_sub):,}')
print(f'  Columns: {list(sample_sub.columns)}')
display(sample_sub.head(3))

print('\n=== song_data.csv ===')
print(f'  Rows   : {len(song_data):,}')
print(f'  Columns: {list(song_data.columns)}')
display(song_data.head(3))

Using data directory: /home/anthony/repos/cmpe256/data/raw/
=== train.csv ===
  Rows   : 4,254,583
  Columns: ['user_id', 'song_id', 'rating']


,user_id,song_id,rating
0,1619409,1003008,10.00
1,1006031,1006608,5.50
2,1376103,1001728,4.25



=== test.csv ===
  Rows   : 1,063,646
  Columns: ['user_id', 'song_id']


,user_id,song_id
0,1553719,1000121
1,1089653,1001182
2,1536350,1008101



=== sample_submission.csv ===
  Rows   : 1,063,646
  Columns: ['RecommendationId', 'rating']


,RecommendationId,rating
0,1553719-1000121,5.37
1,1089653-1001182,5.37
2,1536350-1008101,5.37



=== song_data.csv ===
  Rows   : 198,724
  Columns: ['song_id', 'title', 'release', 'artist_name', 'year']


,song_id,title,release,artist_name,year
0,1191338,Silent Night,Monster Ballads X-Mas,Faster Pussy cat,2003
1,1178322,L'antarctique,Des cobras des tarentules,3 Gars Su'l Sofa,2007
2,1052052,Ethos of Coercion,Descend Into Depravity,Dying Fetus,2009


## Step 3 — Build a Validation Split

Before submitting to Kaggle, always validate your model **locally** using a held-out portion of `train.csv`.  
This gives you a reliable RMSE estimate without wasting Kaggle submissions.

We use an 80/20 split here.

In [4]:
df_train, df_val = train_test_split(train, test_size=0.2, random_state=42)

print(f'Training rows  : {len(df_train):,}')
print(f'Validation rows: {len(df_val):,}')

def rmse(y_true, y_pred):
    """Compute Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

Training rows  : 3,403,666
Validation rows: 850,917


## Step 4 — Baseline Model (Global Mean)

The simplest possible model: predict the **global average rating** for every user-song pair.

This is the official competition baseline. **Your goal is to beat this RMSE.**

In [5]:
# Compute global mean from training split only
global_mean = df_train['rating'].mean()
print(f'Global mean rating: {global_mean:.4f}')

# Predict the same value for every validation pair
val_preds = np.full(len(df_val), global_mean)
baseline_rmse = rmse(df_val['rating'], val_preds)
print(f'Baseline RMSE (validation): {baseline_rmse:.4f}')
print()
print('Beat this score on the public leaderboard to earn ranking points.')

Global mean rating: 5.3703
Baseline RMSE (validation): 1.7316

Beat this score on the public leaderboard to earn ranking points.


## Step 5 — Train on Full Data & Generate Predictions

Once you are happy with your model's validation RMSE, retrain it on the **full** `train.csv` (not just the 80% split) before generating your submission. This uses all available signal.

In [6]:
# Retrain on the full training set
global_mean_full = train['rating'].mean()

# Predict for every row in test.csv
# Replace this line with your own model's predictions
test_preds = np.full(len(test), global_mean_full)

print(f'Predictions shape : {test_preds.shape}')
print(f'Prediction range  : [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'Prediction mean   : {test_preds.mean():.4f}')

Predictions shape : (1063646,)
Prediction range  : [5.370, 5.370]
Prediction mean   : 5.3703


## XGBoost solution

In [7]:
def MFDataHandling(data:pd.DataFrame):
    userIds = data['user_id'].unique()
    songIds = data['song_id'].unique()

    userIdsIndicesMap = {user:i for i,user in enumerate(userIds)}
    songIdsIndicesMap = {song:i for i,song in enumerate(songIds)}
    
    new_data = data.copy()
    new_data['user_id'] = new_data['user_id'].map(userIdsIndicesMap)
    new_data['song_id'] = new_data['song_id'].map(songIdsIndicesMap)
    
    return new_data
mfTrain = MFDataHandling(train)
mfTest = MFDataHandling(test)


In [8]:
mfTrain

,user_id,song_id,rating
0,0,0,10.00
1,1,1,5.50
2,2,2,4.25
3,3,3,9.25
4,4,4,5.00
...,...,...,...
4254578,58870,30323,5.00
4254579,642581,718,5.00
4254580,114140,10871,4.50
4254581,276387,5408,8.75


In [9]:
mfTest

,user_id,song_id
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
...,...,...
1063641,453211,54446
1063642,91572,13019
1063643,190040,36804
1063644,31722,14232


## XGBOOST
### Validation RSME: 1.73157

In [45]:
user_stats = mfTrain.groupby("user_id")['rating'].agg(['mean','count','std']).reset_index()
user_stats.columns = ['user_id','u_mean','u_count','u_std']
song_stats = mfTrain.groupby("song_id")['rating'].agg(['mean','count','std']).reset_index()
song_stats.columns = ['song_id','s_mean','s_count','s_std']
user_stats = user_stats.fillna(0)
song_stats = song_stats.fillna(0)

In [46]:
user_stats = mfTrain.groupby("user_id")['rating'].agg(['mean','count','std']).reset_index()
user_stats.columns = ['user_id','u_mean','u_count','u_std']
song_stats = mfTrain.groupby("song_id")['rating'].agg(['mean','count','std']).reset_index()
song_stats.columns = ['song_id','s_mean','s_count','s_std']
user_stats = user_stats.fillna(0)
song_stats = song_stats.fillna(0)

def get_xgb_features(df,user_stats,song_stats,global_mean):
    df_feat = df.merge(user_stats, on='user_id',how='left')
    df_feat = df_feat.merge(song_stats, on='song_id', how='left')

    df_feat['u_count'] = df_feat['u_count'].fillna(0)
    df_feat['s_count'] = df_feat['s_count'].fillna(0)
    df_feat['u_std'] = df_feat['u_std'].fillna(0)
    df_feat['s_std'] = df_feat['s_std'].fillna(0)

    df_feat['u_mean'] = df_feat['u_mean'].fillna(global_mean)
    df_feat['s_mean'] = df_feat['s_mean'].fillna(global_mean)

    df_feat['u_bias'] = df_feat['u_mean'] - global_mean
    df_feat['s_bias'] = df_feat['s_mean'] - global_mean

    df_feat = df_feat.fillna(global_mean)

    features = ['u_mean', 'u_count', 'u_std', 's_mean', 's_count', 's_std', 'u_bias', 's_bias']
    X = df_feat[features]
    y = df_feat.get('rating',None)

    return X,y
global_mean = mfTrain['rating'].mean()
X_train, y_train = get_xgb_features(mfTrain,user_stats,song_stats,global_mean)
X_val, y_val = get_xgb_features(mfVal,user_stats,song_stats,global_mean)

In [47]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,       
    learning_rate=0.001, 
    max_depth=2,
    base_score=global_mean,
    tree_method='hist',
    eval_metric='rmse',
    early_stopping_rounds=5  
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)

[0]	validation_0-rmse:1.73159
[10]	validation_0-rmse:1.73159
[13]	validation_0-rmse:1.73159


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",np.float64(5.370283585958953)
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",5
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.dataset

## LightGBM
### Best Model so far: Valid rmse: 1.73157

In [14]:
mfTrain

,user_id,song_id,rating
0,0,0,10.00
1,1,1,5.50
2,2,2,4.25
3,3,3,9.25
4,4,4,5.00
...,...,...,...
4254578,58870,30323,5.00
4254579,642581,718,5.00
4254580,114140,10871,4.50
4254581,276387,5408,8.75


In [15]:
import lightgbm as lgb
dtrain = lgb.Dataset(X_train,label=y_train)
dval = lgb.Dataset(X_val,label=y_val,reference=dtrain)

In [16]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,   
    'num_leaves': 31,        
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l2': 10,         
    'verbosity': -1,
    'seed': 42
}
dtrain.set_init_score([5.3703] * len(y_train))
dval.set_init_score([5.3703] * len(y_val))

model_lgb= lgb.train(
    params,
    dtrain,
    valid_sets=[dtrain,dval],
    valid_names=['train','valid'],
    num_boost_round=200,
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=10)
    ]
)

Training until validation scores don't improve for 30 rounds
[10]	train's rmse: 1.71049	valid's rmse: 1.73272
[20]	train's rmse: 1.69136	valid's rmse: 1.73633
[30]	train's rmse: 1.67531	valid's rmse: 1.74176
Early stopping, best iteration is:
[1]	train's rmse: 1.73046	valid's rmse: 1.73157


In [17]:
import torch
torch.cuda.is_available()

True

## Surprise
### Validation RSME:  1.7341

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Learned with Google Gemini.
class SimpleMF(nn.Module):
    def __init__(self,n_users, n_items, n_factors=20):
        super().__init__()
        self.user_factors = nn.Embedding(n_users + 1, n_factors)
        self.item_factors = nn.Embedding(n_items + 1, n_factors)
        self.user_bias = nn.Embedding(n_users + 1,1)
        self.item_bias = nn.Embedding(n_items + 1,1)
        self.offset = 5.3703
        # Initialize weights
        nn.init.normal_(self.user_factors.weight, std=0.01)
        nn.init.normal_(self.item_factors.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

        with torch.no_grad():
            self.user_factors.weight[-1]=0
            self.item_factors.weight[-1]=0
            self.user_bias.weight[-1]=0
            self.item_bias.weight[-1]=0
            
    def forward(self, user, item):

        dot = (self.user_factors(user) * self.item_factors(item)).sum(1,keepdim=True)
        res = dot + self.user_bias(user) + self.item_bias(item) + self.offset
        return res.squeeze()

user_map = {_id:i for i, _id in enumerate(mfTrain['user_id'].unique())}
item_map = {_id:i for i, _id in enumerate(mfTrain['song_id'].unique())}

train_u = torch.LongTensor(mfTrain['user_id'].map(user_map).values)
train_i = torch.LongTensor(mfTrain['song_id'].map(item_map).values)
train_y = torch.FloatTensor(mfTrain['rating'].values)

val_u_raw = mfVal['user_id'].apply(lambda x: user_map.get(x, len(user_map))).values
val_i_raw = mfVal['song_id'].apply(lambda x: item_map.get(x, len(item_map))).values
val_y = torch.FloatTensor(mfVal['rating'].values)

In [29]:
best_rmse = float('inf')

In [34]:
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import gc
# del X_train, X_val
if 'X_train' in locals(): del X_train
if 'X_val' in locals(): del X_val
gc.collect()
# Create data loaders
n_users= len(user_map)
n_items= len(item_map)
training_dataset = TensorDataset(train_u,train_i,train_y)
train_loader = DataLoader(training_dataset, batch_size=4096, shuffle=True, num_workers=2, pin_memory=True)
val_u = torch.LongTensor(val_u_raw)
val_i = torch.LongTensor(val_i_raw)
val_loader = DataLoader(TensorDataset(val_u, val_i,val_y), batch_size=4096, num_workers=2, pin_memory=True)




model = SimpleMF(n_users,n_items,n_factors=150)
# model.load_state_dict(torch.load('best_music_model.pth'))
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=1, factor=0.5)
criterion = nn.MSELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

epochs = 12
for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for u, i, y in train_pbar:
        u, i, y = u.to(device),i.to(device),y.to(device)

        #Forward
        preds = model(u,i)


        #Backward pass
        optimizer.zero_grad()
        loss = criterion(preds,y)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        train_loss += loss.item() * u.size(0)

        train_pbar.set_postfix({"batch_rmse":f"{batch_loss**0.5:.4f}"})

    train_rmse = (train_loss /len(train_loader.dataset))**0.5

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for u, i, y in val_loader:
            u, i, y = u.to(device),i.to(device),y.to(device)
            preds = model(u,i)
            loss = criterion(preds,y)
            val_loss += loss.item() * u.size(0)

    val_rmse = (val_loss / len(val_loader.dataset)) ** 0.5
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_rmse)
    new_lr = optimizer.param_groups[0]['lr']

    if old_lr != new_lr:
        print(f"📉 Learning Rate reduced from {old_lr} to {new_lr}")

    print(f"Epoch {epoch+1} Complete | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

    

/home/anthony/repos/cmpe256/.venv/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:45: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  if warn_imbalance(lambda props: props.total_memory):


Using 2 GPUs!


Epoch 1/12 [Train]: 100%|██████████| 1039/1039 [01:53<00:00,  9.13it/s, batch_rmse=7.8954]


Epoch 1 Complete | Train RMSE: 9.5504 | Val RMSE: 6.7615


Epoch 2/12 [Train]: 100%|██████████| 1039/1039 [01:55<00:00,  9.01it/s, batch_rmse=5.4372]


...

Epoch 10 Complete | Train RMSE: 1.7055 | Val RMSE: 1.7341


Epoch 11/12 [Train]: 100%|██████████| 1039/1039 [01:53<00:00,  9.14it/s, batch_rmse=1.6792]


📉 Learning Rate reduced from 0.001 to 0.0005
Epoch 11 Complete | Train RMSE: 1.7076 | Val RMSE: 1.7341


Epoch 12/12 [Train]: 100%|██████████| 1039/1039 [01:53<00:00,  9.14it/s, batch_rmse=1.7077]


Epoch 12 Complete | Train RMSE: 1.7061 | Val RMSE: 1.7341


In [ ]:
# Check if Val RMSE improved
if val_rmse < best_rmse:
    best_rmse = val_rmse
    
    # Save better model
    model_to_save = model.module if hasattr(model, 'module') else model
    torch.save(model_to_save.state_dict(), 'best_music_model.pth')
    
    print(f"🌟 Saved new best model with Val RMSE: {val_rmse:.4f}")

🌟 Saved new best model with Val RMSE: 1.7341


In [36]:
mf_model = SimpleMF(n_users,n_items,n_factors=150).load_state_dict(torch.load('best_music_model.pth'))
mf_val_preds = []
with torch.no_grad():
    for u, i, y in val_loader:
            u, i, y = u.to(device), i.to(device), y.to(device)
            preds = model(u, i)
            mf_val_preds.extend(preds.cpu().numpy())
mf_val_preds = np.array(mf_val_preds)

In [40]:
lgbm_val_preds = model_lgb.predict(X_val, num_iteration=model_lgb.best_iteration)
lgbm_val_preds = np.array(lgbm_val_preds)


In [48]:
xgb_val_preds = xgb_model.predict(X_val)
xgb_val_preds = np.array(xgb_val_preds)

In [41]:
from sklearn.metrics import mean_squared_error

blended_val_preds = (0.5 * mf_val_preds) +(0.5 * lgbm_val_preds)
blended_val_preds = np.clip(blended_val_preds,1.0,10.0)
final_val_rmse = np.sqrt(mean_squared_error(mfVal['rating'],blended_val_preds))
print(f"--- RESULTS ---")
print(f"MF Alone: 1.7341")
print(f"LGBM Alone: 1.7315")
print(f"Ensemble RMSE: {final_val_rmse:.4f}")


--- RESULTS ---
MF Alone: 1.7341
LGBM Alone: 1.7315
Ensemble RMSE: 3.1958


## 3-Ensemble Model
Consist of the previous 3 trained model that made it under 1.73157 rsme

In [ ]:
# Learned with Google Gemini
def predict_with_fallback(model, u_idx, i_idx, global_mean=5.3703, batch_size=4096):
    """
    Predicts in batches to use the GPU efficiently and handles unknown 
    IDs (mapped to len(user_map)) by returning the global_mean.
    """
    model.eval()
    device = next(model.parameters()).device
    all_preds = np.zeros(len(u_idx))
    
    # Identify unknown users or songs (those we mapped to the 'extra' index)
    # We used n_users/n_items as the unknown index
    n_users = model.module.user_factors.num_embeddings - 1 if hasattr(model, 'module') else model.user_factors.num_embeddings - 1
    n_items = model.module.item_factors.num_embeddings - 1 if hasattr(model, 'module') else model.item_factors.num_embeddings - 1
    
    # Create a mask for rows where the model should provide a prediction
    # If the index is less than our "unknown" index, it's a known entity
    known_mask = (u_idx < n_users) & (i_idx < n_items)
    
    # 1. Fill everything with Global Mean initially
    all_preds[:] = global_mean
    
    # 2. Only run the model on the IDs it actually knows
    known_indices = np.where(known_mask)[0]
    
    for i in range(0, len(known_indices), batch_size):
        batch_idx = known_indices[i : i + batch_size]
        
        u_batch = torch.LongTensor(u_idx[batch_idx]).to(device)
        i_batch = torch.LongTensor(i_idx[batch_idx]).to(device)
        
        with torch.no_grad():
            preds = model(u_batch, i_batch)
            all_preds[batch_idx] = preds.cpu().numpy()
            
    return all_preds

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
# Stack all three models predictions.
X_meta = np.column_stack([mf_val_preds, lgbm_val_preds, xgb_val_preds])
y_meta = mfVal['rating'].values

# Train the meta-model on all models' prediction of 
meta_model = LinearRegression()
meta_model.fit(X_meta, y_meta)

# Check the weights
weights = meta_model.coef_
print(f"Weights: MF={weights[0]:.4f}, LGBM={weights[1]:.4f}, XGB={weights[2]:.4f}")

# Final Score
final_preds = meta_model.predict(X_meta)
final_rmse = np.sqrt(mean_squared_error(y_meta, np.clip(final_preds, 1, 10)))
print(f"--- 3-Model Ensemble RMSE: {final_rmse:.5f} ---")

Weights: MF=0.2216, LGBM=1.1268, XGB=-3.3547
--- 3-Model Ensemble RMSE: 1.73141 ---


In [ ]:
# Learned with Google Gemini.
def ensemble_model_prediction(test_df, X_test_gbdt, mf_model, lgb_model, xgb_model, meta_model, user_map, item_map):
    """
    Returns a single NumPy array of the final ensemble predictions.
    """
    # 1. MF Predictions (Handling unknowns and GPU batches)
    u_idx = test_df['user_id'].apply(lambda x: user_map.get(x, len(user_map))).values
    i_idx = test_df['song_id'].apply(lambda x: item_map.get(x, len(item_map))).values
    
    mf_preds = predict_with_fallback(mf_model, u_idx, i_idx, global_mean=5.3703)

    # 2. GBDT Predictions
    lgbm_preds = lgb_model.predict(X_test_gbdt)
    xgb_preds = xgb_model.predict(X_test_gbdt)

    # 3. Meta-Stacking
    # Note: Ensure the order [mf, lgbm, xgb] matches your meta_model training!
    X_meta = np.column_stack([mf_preds, lgbm_preds, xgb_preds])
    
    final_preds = meta_model.predict(X_meta)
    
    # 4. Final Clipping
    return np.clip(final_preds, 1.0, 10.0)


In [60]:
# Retrain on the full training set
global_mean_full = train['rating'].mean()

# Predict for every row in test.csv
# Replace this line with your own model's predictions
X_test, _ = get_xgb_features(mfTest, user_stats, song_stats, global_mean)
# test_preds = np.clip(model_lgb.predict(X_test),1.0,10.0)
# test_preds = model_lgb.predict(X_test)
# 1. Re-initialize the model structure
n_users = len(user_map)
n_items = len(item_map)
mf_model = SimpleMF(n_users, n_items, n_factors=150)

# 2. Load the weights (Note: DO NOT do mf_model = mf_model.load_state_dict)
mf_model.load_state_dict(torch.load('best_music_model.pth'))

# 3. Move to GPU/DataParallel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    mf_model = nn.DataParallel(mf_model)
mf_model.to(device)

# Now your call to ensemble_model_prediction will work!
test_preds = ensemble_model_prediction(
    mfTest, X_test, 
    mf_model=mf_model, 
    lgb_model=model_lgb, 
    xgb_model=xgb_model, 
    meta_model=meta_model, 
    user_map=user_map, 
    item_map=item_map
)
# model = SimpleMF(n_users,n_items,n_factors=150).load_state_dict(torch.load('best_music_model.pth'))
# test_preds = ensemble_model_prediction(mfTest,X_test,mf_model=mf_model,lgb_model=model_lgb,xgb_model=xgb_model,meta_model=meta_model,user_map=user_map,item_map=item_map)

print(f'Predictions shape : {test_preds.shape}')
print(f'Prediction range  : [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'Prediction mean   : {test_preds.mean():.4f}')

/home/anthony/repos/cmpe256/.venv/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:45: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  if warn_imbalance(lambda props: props.total_memory):


Predictions shape : (1063646,)
Prediction range  : [5.170, 5.699]
Prediction mean   : 5.3705


## Step 6 — Build & Validate the Submission File

The submission format requires:
- Column `RecommendationId` — formatted as `user_id-song_id`
- Column `rating` — your predicted rating, clipped to [1.0, 10.0]

We validate the format against `sample_submission.csv` before saving.

In [62]:
# ── Build submission dataframe ────────────────────────────────────────────────
submission = pd.DataFrame({
    'RecommendationId': test['user_id'].astype(str) + '-' + test['song_id'].astype(str),
    'rating': np.clip(test_preds, 1.0, 10.0)   # always clip to valid range
})

# ── Format validation ─────────────────────────────────────────────────────────
print('=== Submission Validation ===')

# 1. Column names match
assert list(submission.columns) == list(sample_sub.columns), \
    f'Column mismatch: got {list(submission.columns)}, expected {list(sample_sub.columns)}'
print('✓ Column names match sample_submission.csv')

# 2. Row count matches test.csv
assert len(submission) == len(test), \
    f'Row count mismatch: got {len(submission)}, expected {len(test)}'
print(f'✓ Row count matches test.csv ({len(submission):,} rows)')

# 3. No missing values
assert submission.isnull().sum().sum() == 0, 'Submission contains NaN values!'
print('✓ No missing values')

# 4. Ratings within valid range
assert submission['rating'].between(1.0, 10.0).all(), 'Some ratings are outside [1.0, 10.0]!'
print('✓ All ratings within valid range [1.0, 10.0]')

# 5. RecommendationId format looks correct
id_check = submission['RecommendationId'].str.match(r'^\d+-\d+$').all()
assert id_check, 'Some RecommendationId values have unexpected format!'
print('✓ RecommendationId format is correct (user_id-song_id)')

print()
print('All checks passed. Ready to submit!')
print()
display(submission.head(5))

=== Submission Validation ===
✓ Column names match sample_submission.csv
✓ Row count matches test.csv (1,063,646 rows)
✓ No missing values
✓ All ratings within valid range [1.0, 10.0]
✓ RecommendationId format is correct (user_id-song_id)

All checks passed. Ready to submit!



,RecommendationId,rating
0,1553719-1000121,5.412271
1,1089653-1001182,5.367202
2,1536350-1008101,5.440466
3,1619708-1017052,5.372748
4,1048204-1013148,5.523264


## Step 7 — Save & Download the Submission File

In [63]:
OUTPUT_FILE = DATA_DIR + 'solution.csv'
submission.to_csv(OUTPUT_FILE, index=False)
print(f'Saved to: {OUTPUT_FILE}')

Saved to: /home/anthony/repos/cmpe256/data/raw/solution.csv


## Step 8 — Check if solution file format matching with sample solution file

In [64]:
sub    = pd.read_csv(OUTPUT_FILE)
sample = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print('=== Your Submission ===')
print('Columns:', sub.columns.tolist())
print('Rows:   ', len(sub))
print('dtypes:')
print(sub.dtypes)
print()
print('Preview:')
print(sub.head(3))

print()
print('=== Sample Submission ===')
print('Columns:', sample.columns.tolist())
print('Rows:   ', len(sample))
print('dtypes:')
print(sample.dtypes)
print()
print('Preview:')
print(sample.head(3))

print()
print('=== Validation Checks ===')
print('Column names match:       ', sub.columns.tolist() == sample.columns.tolist())
print('Row counts match:         ', len(sub) == len(sample))
print('No NaN in rating:         ', sub[sub.columns[1]].isna().sum() == 0)
print('Ratings in range [1, 10]: ', sub[sub.columns[1]].between(1, 10).all())
print('IDs match sample:         ', set(sub[sub.columns[0]]) == set(sample[sample.columns[0]]))
print('ID format correct:        ', sub[sub.columns[0]].str.contains(r'^\d+-\d+$').all())

=== Your Submission ===
Columns: ['RecommendationId', 'rating']
Rows:    1063646
dtypes:
RecommendationId        str
rating              float64
dtype: object

Preview:
  RecommendationId    rating
0  1553719-1000121  5.412271
1  1089653-1001182  5.367202
2  1536350-1008101  5.440466

=== Sample Submission ===
Columns: ['RecommendationId', 'rating']
Rows:    1063646
dtypes:
RecommendationId        str
rating              float64
dtype: object

Preview:
  RecommendationId  rating
0  1553719-1000121    5.37
1  1089653-1001182    5.37
2  1536350-1008101    5.37

=== Validation Checks ===
Column names match:        True
Row counts match:          True
No NaN in rating:          True
Ratings in range [1, 10]:  True
IDs match sample:          True
ID format correct:         True


## Step 9 — Submit to kaggle directly from google colab (or upload solution csv manually)

In [85]:
!kaggle competitions submit \
    -c cmpe-256-song-recommender-system33 \
    -f {OUTPUT_FILE} \
    -m "baseline kaggle competition submission"

print('Submitted - check leaderboard at:')
print('https://www.kaggle.com/competitions/cmpe-256-song-recommender-system33/leaderboard')

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload
Submitted - check leaderboard at:
https://www.kaggle.com/competitions/cmpe-256-song-recommender-system33/leaderboard
